# Demo Machine Translation Indonesia → Jawa dengan Pilihan Tingkat Tutur

Notebook ini menjalankan demo **Gradio** dengan link publik `*.gradio.live` langsung dari Kaggle.

**Model:** M2M100-418M hasil fine-tuning pada dataset Unggah-Ungguh (3 tingkat: Ngoko, Krama, Krama Alus),
ditambah rerank 8 kandidat beam dengan classifier tingkat tutur `w11wo/javanese-roberta-small`.

## Cara memakai di Kaggle
1. Kaggle → **Create → New Notebook** → menu **File → Import Notebook** → unggah file ini.
2. Panel kanan **Input → Add Input** → tab **Notebooks** → cari dan tambahkan output notebook
   `unggah-ungguh-final-3lv-v2` (berisi weights model dan reranker).
3. **Settings → Internet: On** (wajib, untuk instalasi Gradio dan link publik).
   Accelerator boleh **None** (CPU, sekitar 3–10 detik per kalimat) atau **GPU T4** (lebih cepat).
4. **Run All**. Tunggu sel "Jalankan Gradio" mencetak link `https://xxxx.gradio.live`, lalu buka link itu.
5. Link hidup selama sesi notebook berjalan (link Gradio sendiri kedaluwarsa sesudah 1 minggu).
   Untuk mematikan demo: tombol **Stop / Interrupt** atau matikan sesi.

Catatan: lisensi dataset CC BY-NC 4.0, jadi demo ini hanya untuk keperluan riset dan kuliah.

## 1. Instalasi pustaka

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio", "sentencepiece"], check=False)
import gradio as gr, transformers, torch
print("gradio", gr.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)

## 2. Mencari folder model di input Kaggle

In [ ]:
import glob, os
from pathlib import Path

kandidat = [p for p in glob.glob("/kaggle/input/**/demo_config.json", recursive=True) if "/reranker/" not in p]
if not kandidat:
    raise FileNotFoundError(
        "demo_config.json tidak ditemukan di /kaggle/input. Tambahkan output notebook "
        "'unggah-ungguh-final-3lv-v2' lewat Add Input → Notebooks, lalu jalankan ulang sel ini."
    )
MODEL_DIR = Path(os.path.dirname(kandidat[0]))
print("Folder model:", MODEL_DIR)
print(sorted(p.name for p in MODEL_DIR.iterdir()))

## 3. Memuat model, tokenizer, dan reranker

In [ ]:
import json, time
import torch
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, AutoTokenizer

CFG = json.loads((MODEL_DIR / "demo_config.json").read_text(encoding="utf-8"))
LEVELS = CFG["levels"]
TAGS = CFG["tags"]
LABEL = {"ngoko": "Ngoko", "ngoko_alus": "Ngoko Alus", "krama": "Krama", "krama_alus": "Krama Alus"}
NOTE = {
    "ngoko": "akrab, sebaya, atau ke yang lebih muda",
    "ngoko_alus": "akrab tetapi menghormati orang yang dibicarakan",
    "krama": "sopan ke orang yang belum akrab atau lebih tua",
    "krama_alus": "paling halus, ke orang yang sangat dihormati",
}
MAXLEN = 64
MAX_CHARS = 300
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL_DIR, src_lang=CFG["src_lang"], tgt_lang=CFG["tgt_lang"])
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
BOS = tok.get_lang_id(CFG["tgt_lang"]) if hasattr(tok, "get_lang_id") else tok.convert_tokens_to_ids(CFG["tgt_lang"])

RERANK = CFG.get("decode") == "rerank"
if RERANK:
    NBEST, LAM = int(CFG["nbest"]), float(CFG["rerank_lambda"])
    rr_dir = MODEL_DIR / CFG.get("reranker_dir", "reranker")
    rr_tok = AutoTokenizer.from_pretrained(rr_dir)
    rr_mod = AutoModelForSequenceClassification.from_pretrained(rr_dir).float().to(DEVICE).eval()

print("Perangkat:", DEVICE, "| tingkat:", LEVELS, "| rerank:", RERANK,
      f"(kandidat {NBEST}, lambda {LAM})" if RERANK else "")

### 3b. Memuat model arah sebaliknya (Jawa → Indonesia)

Model M2M100-418M (`jv` → `id`) dari output notebook `unggah-ungguh-reverse-jv-id-v1`, lengan `semua`
(Unggah-Ungguh + data eksternal, 15 epoch, chrF++ valid 55,72). Input kalimat Jawa tingkat apa pun, tanpa tag.

In [ ]:
REV_ARM = "semua"
rev_cand = glob.glob(f"/kaggle/input/**/model_{REV_ARM}/config.json", recursive=True)
if not rev_cand:
    raise FileNotFoundError(
        f"model_{REV_ARM} tidak ditemukan. Tambahkan output notebook 'unggah-ungguh-reverse-jv-id-v1' "
        "lewat Add Input → Notebooks, lalu jalankan ulang sel ini."
    )
REV_DIR = Path(os.path.dirname(rev_cand[0]))
rev_tok = AutoTokenizer.from_pretrained(REV_DIR, src_lang="jv", tgt_lang="id")
rev_model = AutoModelForSeq2SeqLM.from_pretrained(REV_DIR).float().to(DEVICE).eval()
REV_BOS = rev_tok.get_lang_id("id")
REV_BEAMS = 5
print("Model Jawa → Indonesia:", REV_DIR)

## 4. Fungsi terjemahan

Input diberi tag tingkat (mis. `<krama_alus> Bapak sudah pulang.`). Dengan rerank, model membuat 8 kandidat beam,
lalu memilih kandidat dengan skor = rata-rata log-prob model + lambda × log P(tingkat diminta) dari reranker.

In [ ]:
def _beam(sources, k):
    enc = tok(sources, return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, forced_bos_token_id=BOS, num_beams=max(k, 4), num_return_sequences=k,
                             max_new_tokens=MAXLEN)
    return tok.batch_decode(out, skip_special_tokens=True)


def _seq_scores(sources, cands):
    # Rata-rata log-prob token kandidat menurut model (teacher forcing).
    enc = tok(sources, text_target=cands, return_tensors="pt", padding=True, truncation=True,
              max_length=MAXLEN).to(DEVICE)
    labels = enc.pop("labels")
    labels[labels == tok.pad_token_id] = -100
    with torch.no_grad():
        logits = model(**enc, labels=labels).logits
    nll = F.cross_entropy(logits.transpose(1, 2), labels, ignore_index=-100, reduction="none")
    m = (labels != -100).float()
    return (-(nll * m).sum(1) / m.sum(1).clamp(min=1)).cpu()


def _level_logprobs(sents):
    enc = rr_tok(sents, return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(DEVICE)
    with torch.no_grad():
        return F.log_softmax(rr_mod(**enc).logits, -1).cpu()


def terjemahkan(text, levels):
    sources = [f"{TAGS[l]} {text}" for l in levels]
    if not RERANK:
        return _beam(sources, 1)
    cands = _beam(sources, NBEST)
    score = _seq_scores([s for s in sources for _ in range(NBEST)], cands)
    lp = _level_logprobs(cands)
    outs = []
    for i, l in enumerate(levels):
        sl = slice(i * NBEST, (i + 1) * NBEST)
        total = score[sl] + LAM * lp[sl, LEVELS.index(l)]
        outs.append(cands[sl][int(total.argmax())])
    return outs


def terjemahkan_jv_id(text):
    enc = rev_tok([text], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(DEVICE)
    with torch.no_grad():
        out = rev_model.generate(**enc, forced_bos_token_id=REV_BOS, num_beams=REV_BEAMS, max_new_tokens=MAXLEN)
    return rev_tok.batch_decode(out, skip_special_tokens=True)[0]

## 5. Uji cepat tanpa antarmuka (sekaligus pemanasan model)

In [ ]:
CONTOH = [
    "Itu, kakek disuruh mandi terlebih dahulu!",
    "Mbak Srini makan pecel dengan ajang pincuk.",
    "Bapak sudah pulang dari kantor.",
    "Saya mau pergi ke pasar membeli sayur.",
    "Ibu sedang tidur di kamar.",
]
for kalimat in CONTOH[:3]:
    t0 = time.time()
    hasil = terjemahkan(kalimat, LEVELS)
    print(f"\n{kalimat}   ({time.time() - t0:.1f} detik)")
    for l, o in zip(LEVELS, hasil):
        print(f"  {LABEL[l]:<11} {o}")

CONTOH_JV = [
    "Bapak sampun kondur saking kantor.",
    "Aku arep lunga menyang pasar tuku sayur.",
    "Ibu nembe sare wonten kamar.",
    "Simbah dipunaturi siram rumiyin.",
    "Kowe wis mangan durung?",
]
print("\nJawa → Indonesia")
for kalimat in CONTOH_JV[:3]:
    t0 = time.time()
    print(f"  {kalimat}  →  {terjemahkan_jv_id(kalimat)}   ({time.time() - t0:.1f} detik)")

## 6. Membangun antarmuka Gradio

In [ ]:
def _clean(text, pesan_kosong="Tulis kalimat bahasa Indonesia terlebih dahulu."):
    text = " ".join((text or "").split())
    if not text:
        raise gr.Error(pesan_kosong)
    if len(text) > MAX_CHARS:
        raise gr.Error(f"Kalimat terlalu panjang (maksimum {MAX_CHARS} karakter). Model dilatih pada kalimat pendek.")
    return text


def translate_one(text, level_label):
    text = _clean(text)
    level = next(k for k, v in LABEL.items() if v == level_label)
    t0 = time.time()
    out = terjemahkan(text, [level])[0]
    return out, f"{LABEL[level]}: {NOTE[level]} · {time.time() - t0:.1f} detik"


def translate_all(text):
    text = _clean(text)
    t0 = time.time()
    outs = terjemahkan(text, LEVELS)
    rows = [[LABEL[l], o] for l, o in zip(LEVELS, outs)]
    return rows, f"{len(LEVELS)} tingkat · {time.time() - t0:.1f} detik"


def _eval_note():
    t = CFG.get("test")
    if not t:
        return ""
    num = lambda x, d: f"{x:.{d}f}".replace(".", ",")
    per = " · ".join(f"{LABEL[l]} {num(t['control_per_level'][l], 2)}" for l in LEVELS)
    chrf, bleu = t["chrF++/BLEU"]["all"]
    return (f"- Hasil evaluasi (test set, seed {CFG.get('seed', 42)}): chrF++ {num(chrf, 1)} · BLEU {num(bleu, 1)} · "
            f"ketepatan tingkat {num(t['control_acc'], 2)} ({per}).\n")


def translate_rev(text):
    text = _clean(text, "Tulis kalimat bahasa Jawa terlebih dahulu.")
    t0 = time.time()
    return terjemahkan_jv_id(text), f"{time.time() - t0:.1f} detik"


LEVEL_NAMES = ", ".join(LABEL[l] for l in LEVELS)

with gr.Blocks(title="MT Indonesia ⇄ Jawa Tingkat Tutur") as demo:
    gr.Markdown("# Penerjemah Indonesia ⇄ Jawa dengan Pilihan Tingkat Tutur")

    with gr.Tab("Indonesia → Jawa"):
        gr.Markdown(
            f"Ketik kalimat bahasa Indonesia, lalu pilih tingkat tutur bahasa Jawa ({LEVEL_NAMES}). "
            "Model: **M2M100-418M** hasil fine-tuning pada dataset **Unggah-Ungguh** dengan tag tingkat tutur"
            + (", ditambah *rerank* kandidat dengan classifier tingkat tutur." if RERANK else ".")
        )
        with gr.Row():
            with gr.Column(scale=3):
                src = gr.Textbox(label="Kalimat bahasa Indonesia", lines=3, max_lines=6,
                                 placeholder="Contoh: Bapak sudah pulang dari kantor.")
                level = gr.Radio([LABEL[l] for l in LEVELS], value=LABEL[LEVELS[-1]], label="Tingkat tutur")
                with gr.Row():
                    btn_one = gr.Button("Terjemahkan", variant="primary")
                    btn_all = gr.Button(f"Bandingkan {len(LEVELS)} tingkat")
                gr.Examples(CONTOH, inputs=src, label="Contoh kalimat")
            with gr.Column(scale=4):
                out_one = gr.Textbox(label="Terjemahan bahasa Jawa", lines=3, interactive=False)
                info = gr.Markdown()
                out_all = gr.Dataframe(headers=["Tingkat", "Terjemahan"], label="Semua tingkat",
                                       wrap=True, interactive=False)

    with gr.Tab("Jawa → Indonesia"):
        gr.Markdown(
            "Ketik kalimat bahasa Jawa (Ngoko, Krama, atau Krama Alus). "
            "Model: **M2M100-418M** hasil fine-tuning arah Jawa → Indonesia pada Unggah-Ungguh ditambah data eksternal."
        )
        with gr.Row():
            with gr.Column(scale=3):
                rev_src = gr.Textbox(label="Kalimat bahasa Jawa", lines=3, max_lines=6,
                                     placeholder="Contoh: Bapak sampun kondur saking kantor.")
                rev_btn = gr.Button("Terjemahkan", variant="primary")
                gr.Examples(CONTOH_JV, inputs=rev_src, label="Contoh kalimat")
            with gr.Column(scale=4):
                rev_out = gr.Textbox(label="Terjemahan bahasa Indonesia", lines=3, interactive=False)
                rev_info = gr.Markdown()

    btn_one.click(translate_one, [src, level], [out_one, info])
    src.submit(translate_one, [src, level], [out_one, info])
    btn_all.click(translate_all, [src], [out_all, info])
    rev_btn.click(translate_rev, [rev_src], [rev_out, rev_info])
    rev_src.submit(translate_rev, [rev_src], [rev_out, rev_info])

    gr.Markdown(
        "### Catatan\n"
        + _eval_note()
        + "- Tingkat **Krama Alus** masih paling sering meleset: permintaan Krama Alus kadang menghasilkan Krama biasa.\n"
        "- Arah **Jawa → Indonesia** masih sering salah makna (kata isi diganti kata lain); chrF++ valid 55,72, belum diuji di test set.\n"
        "- Model dilatih pada kalimat pendek (rata-rata 8 kata). "
        "Kalimat panjang, istilah teknis, atau dialek daerah bisa diterjemahkan kurang tepat.\n"
        "- Output belum diperiksa penutur Jawa; jangan dipakai untuk komunikasi penting tanpa pengecekan.\n"
        "- Dataset utama: Unggah-Ungguh (Farhansyah dkk., ACL 2025), lisensi CC BY-NC 4.0. Demo ini untuk keperluan riset/kuliah."
    )
print("Antarmuka siap.")

## 7. Jalankan Gradio dengan link publik

Link `https://xxxx.gradio.live` dicetak di bawah sel ini. Buka di browser (bisa juga dari HP).
Isi `NTFY_TOPIC` hanya jika notebook dijalankan lewat **Save Version** (mode batch, output sel tidak terlihat langsung):
link akan dikirim ke `https://ntfy.sh/<topik>`. Pakai topik acak yang sulit ditebak.

In [ ]:
import urllib.request

NTFY_TOPIC = ""  # opsional, mis. "mtjawa-<string acak>"

try:
    demo.close()  # kalau sel ini dijalankan ulang, matikan server lama dulu
except Exception:
    pass
demo.queue(max_size=16).launch(share=True, prevent_thread_lock=True, show_error=True)
SHARE_URL = demo.share_url
print("=" * 60)
print("LINK DEMO:", SHARE_URL)
print("=" * 60, flush=True)

if NTFY_TOPIC and SHARE_URL:
    try:
        urllib.request.urlopen(urllib.request.Request(f"https://ntfy.sh/{NTFY_TOPIC}",
                                                      data=str(SHARE_URL).encode(), method="POST"), timeout=30)
        print("Link terkirim ke ntfy topik", NTFY_TOPIC)
    except Exception as e:
        print("ntfy gagal:", e)

## 8. Menjaga demo tetap hidup

Di mode interaktif, sel ini tidak wajib: server Gradio tetap jalan selama sesi notebook aktif.
Di mode **Save Version** (batch), sel ini mencegah notebook selesai sehingga link tetap hidup
(maksimal sekitar 11,5 jam, batas sesi Kaggle). Hentikan dengan tombol **Interrupt**.

In [ ]:
JAM = 11.5
t0 = time.time()
try:
    while time.time() - t0 < JAM * 3600:
        time.sleep(60)
except KeyboardInterrupt:
    print("Dihentikan.")
finally:
    demo.close()
    print("Server Gradio dimatikan.")